# DPO Alignment Notebook
## Stage 3: Direct Preference Optimization

This notebook performs DPO alignment using preference data to improve answer quality.

## Step 1: Install Required Libraries

In [ ]:
!pip install -q torch transformers datasets peft bitsandbytes accelerate unsloth[colab-new] trl -U

## Step 2: Import Libraries

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from datasets import Dataset
from peft import LoraConfig, get_peft_model
from trl import DPOTrainer
import json
import os

print(f"GPU Available: {torch.cuda.is_available()}")
print(f"GPU Name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

## Step 3: Load Preference Dataset

In [ ]:
# Load preference dataset
pref_dataset_path = 'course-doubt-assistant/data/preference_dataset.jsonl'

data = []
with open(pref_dataset_path, 'r', encoding='utf-8') as f:
    for line in f:
        data.append(json.loads(line))

print(f"Loaded {len(data)} preference examples")
print(f"\nFirst example:")
print(f"  Prompt: {data[0]['prompt']}")
print(f"  Chosen: {data[0]['chosen'][:80]}...")
print(f"  Rejected: {data[0]['rejected'][:80]}...")

## Step 4: Create Dataset Object

In [ ]:
# Create dataset for DPO
dataset = Dataset.from_dict({
    'prompt': [d['prompt'] for d in data],
    'chosen': [d['chosen'] for d in data],
    'rejected': [d['rejected'] for d in data]
})

print(f"Dataset size: {len(dataset)}")
print(f"Columns: {dataset.column_names}")

## Step 5: Load Model and Tokenizer

In [ ]:
# Model configuration
MODEL_NAME = 'unsloth/tinyllama-bnb-4bit'

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

# Load SFT model (from previous stage)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map='auto',
    torch_dtype=torch.float16
)

print(f"Model loaded: {MODEL_NAME}")

## Step 6: Configure LoRA for DPO

In [ ]:
# LoRA Configuration for DPO
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'v_proj'],
    modules_to_save=['lm_head']
)

# Apply LoRA
model = get_peft_model(model, lora_config)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable_params:,}")
print(f"Trainable %: {100 * trainable_params / total_params:.2f}%")

## Step 7: Configure DPO Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir='./outputs/dpo_aligned',
    num_train_epochs=2,                    # Fewer epochs for DPO
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-5,                    # Very low LR for preference learning
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=5,
    save_steps=25,
    save_total_limit=2,
    gradient_accumulation_steps=4,
    optim='adamw_8bit',
    seed=42,
    report_to=[],
    max_steps=500                         # DPO typically needs fewer steps
)

print("DPO training arguments configured")

## Step 8: Initialize DPO Trainer

In [ ]:
# Initialize DPO Trainer
dpo_trainer = DPOTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
    beta=0.1,                             # DPO temperature
    loss_type='sigmoid'                   # DPO loss type
)

print("DPO Trainer initialized")

## Step 9: Train Model with DPO

In [ ]:
print("Starting DPO alignment training...")
train_result = dpo_trainer.train()
print(f"DPO training loss: {train_result.training_loss:.4f}")

## Step 10: Save DPO-Aligned Model

In [ ]:
# Save DPO-aligned adapter
dpo_adapter_path = './models/dpo_aligned_adapter'
os.makedirs(dpo_adapter_path, exist_ok=True)
model.save_pretrained(dpo_adapter_path)
tokenizer.save_pretrained(dpo_adapter_path)

print(f"DPO adapter saved to {dpo_adapter_path}")

## Step 11: Test DPO-Aligned Model

In [ ]:
def generate_response(question, max_length=150):
    prompt = f"### Instruction:
{question}
### Response:
"
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    outputs = model.generate(
        inputs.input_ids,
        max_length=max_length,
        temperature=0.5,
        top_p=0.9,
        do_sample=True
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test questions
test_questions = [
    'What is supervised learning?',
    'How does backpropagation work?',
    'Explain cross-validation'
]

print("Testing DPO-aligned model:\n")
for q in test_questions:
    print(f"Question: {q}")
    print(f"Answer: {generate_response(q)}")
    print()